# Solution 3 — Gradient Boosting (sklearn `GradientBoostingRegressor`)

Tree ensembles handle non-linearities and interactions automatically and usually provide a strong tabular baseline. The price effect is curved and promo interacts with weekend — both are things trees pick up without explicit feature engineering.

We still include a few time features (Fourier seasonality, trend) because trees can't extrapolate a linear trend beyond the training range on their own.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

sns.set_theme(style='whitegrid')
DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['date'])
test  = pd.read_csv(DATA_DIR / 'test.csv',  parse_dates=['date'])
train.head()

## 1. EDA — focus on structure trees will exploit

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
train.groupby('dow')['sales'].mean().plot.bar(ax=axes[0])
axes[0].set_title('Mean sales by day of week')

# Moving average to reveal trend + yearly wave
ma = train.set_index('date')['sales'].rolling(30, center=True).mean()
axes[1].plot(ma.index, ma.values)
axes[1].set_title('30-day moving avg (trend + yearly wave)')

sns.scatterplot(data=train, x='price', y='sales', hue='promo', ax=axes[2], s=12)
axes[2].set_title('Price vs sales by promo')
plt.tight_layout(); plt.show()

In [ ]:
# Pairplot-style view of the main numeric features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.violinplot(data=train, x='dow', y='sales', hue='promo', split=True, ax=axes[0])
axes[0].set_title('Promo lift across dow')
sns.histplot(data=train, x='price', hue='promo', ax=axes[1], kde=True)
axes[1].set_title('Price distribution by promo flag')
plt.tight_layout(); plt.show()

## 2. Features
Minimal: raw columns + Fourier terms + linear trend. Let the trees handle the rest.

In [ ]:
def make_features(df):
    X = pd.DataFrame(index=df.index)
    X['t'] = df['t']
    X['dow'] = df['dow']
    X['price'] = df['price']
    X['promo'] = df['promo']
    angle = 2 * np.pi * df['t'] / 365.25
    X['yr_sin'] = np.sin(angle); X['yr_cos'] = np.cos(angle)
    return X

cutoff = len(train) - 90
tr, va = train.iloc[:cutoff], train.iloc[cutoff:]
X_tr, X_va = make_features(tr), make_features(va)

## 3. Train & validate
Compare a Random Forest baseline with a tuned Gradient Boosting model.

In [ ]:
def score(y, p):
    return {'rmse': float(np.sqrt(mean_squared_error(y, p))),
            'mape': float(mean_absolute_percentage_error(y, p))}

rf = RandomForestRegressor(n_estimators=400, max_depth=None, min_samples_leaf=3,
                           random_state=0, n_jobs=-1).fit(X_tr, tr['sales'])
gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=3,
                               subsample=0.8, random_state=0).fit(X_tr, tr['sales'])

pred_rf, pred_gb = rf.predict(X_va), gb.predict(X_va)
print('Random Forest     ', score(va['sales'], pred_rf))
print('Gradient Boosting ', score(va['sales'], pred_gb))

In [ ]:
# Visualize validation fit
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(va['date'], va['sales'].values, label='actual', lw=1.2)
ax.plot(va['date'], pred_rf, label='Random Forest', lw=1, alpha=0.7)
ax.plot(va['date'], pred_gb, label='Gradient Boosting', lw=1.2)
ax.legend(); ax.set_title('Validation: last 90 days of train'); plt.show()

In [ ]:
# Feature importance from GB
imp = pd.Series(gb.feature_importances_, index=X_tr.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 3.5))
imp.plot.barh(ax=ax); ax.set_title('GB feature importance'); plt.show()

In [ ]:
# Residual diagnostics
res = va['sales'].values - pred_gb
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].scatter(pred_gb, res, s=10); axes[0].axhline(0, color='k', lw=0.8)
axes[0].set_xlabel('predicted'); axes[0].set_ylabel('residual'); axes[0].set_title('Residuals vs predicted')
sns.histplot(res, kde=True, ax=axes[1]); axes[1].set_title('Residual distribution')
plt.tight_layout(); plt.show()

## 4. Fit on all train, predict test

In [ ]:
final = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=3,
                                  subsample=0.8, random_state=0).fit(make_features(train), train['sales'])
pred_test = final.predict(make_features(test))

out = pd.DataFrame({'date': test['date'], 'sales_pred': pred_test.round(2)})
out.to_csv('predictions_gbm.csv', index=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train['date'], train['sales'], lw=0.6, label='train')
ax.plot(test['date'], pred_test, lw=1.2, color='red', label='GB forecast')
ax.legend(); ax.set_title('Gradient boosting forecast for next 90 days'); plt.show()
out.head()

## Notes
- Pure tree models can't extrapolate a linear trend — the `t` feature gives them a handle on it but they still cap at the highest observed value. For a longer horizon, detrending the target and adding back the trend post-hoc would be more robust.
- Random Forest is included as a sanity check; gradient boosting usually wins on small tabular problems thanks to its bias-variance profile.